# Feature Selection
The purpose of this script is to isolate the features to be used in my model for training, validation, and test

## GITHUB LINK:
https://github.com/Owenx25/UT_AiInHealthcare/tree/main/HIGH_RISK_PROJECT

In [2]:
import pandas as pd
import numpy as np
import os

DATA_PATH = "./data/"

In [3]:
visits_df = pd.read_csv(os.path.join(DATA_PATH, "visits.csv"))
meds_df = pd.read_csv(os.path.join(DATA_PATH, "meds.csv"))
pmh_df = pd.read_csv(os.path.join(DATA_PATH, "pmh.csv"))

In [4]:
split = {
    "random": {
        "train": pd.read_csv(os.path.join(DATA_PATH, "split_random_train.csv"), header=None),
        "val": pd.read_csv(os.path.join(DATA_PATH, "split_random_val.csv"), header=None),
        "test": pd.read_csv(os.path.join(DATA_PATH, "split_random_test.csv"), header=None),
    },
    "chrono": {
        "train": pd.read_csv(os.path.join(DATA_PATH, "split_chrono_train.csv"), header=None),
        "val": pd.read_csv(os.path.join(DATA_PATH, "split_chrono_val.csv"), header=None),
        "test": pd.read_csv(os.path.join(DATA_PATH, "split_chrono_test.csv"), header=None),
    },
}

SPLIT_TYPE = 'random'
SPLIT_GROUP = 'test'
split = split[SPLIT_TYPE][SPLIT_GROUP]

In [6]:
def filter_by_split(split_df, df):
    """
    Filter the dataframe based on the split provided.
    
    :param split: The split to filter by (train, val, test)
    :param df: The dataframe to filter
    :return: Filtered dataframe
    """
    
    # for each MRN in the split, get the corresponding rows in the dataframe
    # and return the filtered dataframe
    split_mrns = split_df[0]
    filtered_df = df[df["MRN"].isin(split_mrns)]
    return filtered_df.reset_index(drop=True)

In [7]:
def one_hot_encode(df, column):
    """
    One-hot encode a specified column in the dataframe.
    
    :param df: The dataframe to encode
    :param column: The column to one-hot encode
    :return: Dataframe with one-hot encoded column
    """
    df[column] = df[column].str.replace(" ", "_")
    # one-hot encode the specified column
    one_hot = pd.get_dummies(df[column], prefix=column)
    one_hot = one_hot.astype(int)
    
    return one_hot

## Raw Features

In [2142]:
# any categorical features here will be one-hot encoded
# more advanced columns like diagnosis or meds will be 
# converted to embeddings later on
raw_features = {
    "visits": [
        "CSN",
        "Visit_no",
        "Age",
        "Triage_Temp",
        "Triage_HR",
        "Triage_RR",
        "Triage_SpO2",
        "Triage_SBP",
        "Triage_DBP",
    ],
}
raw_features_categorical = {
    "visits": [
        "Race",
        "Ethnicity",
        "Means_of_arrival",
        "Gender",
        "Payor_class",
    ]
}

In [2143]:
def extract_raw_features(split_df, feature_type, raw_features_df):
    """
    Extracts the raw features from the dataframe.
    """
    new_features_df = pd.DataFrame()
    split_df = filter_by_split(split_df, raw_features_df)
    
    if feature_type in raw_features:
        for feature in raw_features[feature_type]:
            # keep the numerical features as is
            new_features_df[feature] = split_df[feature]
    if feature_type in raw_features_categorical:
        for feature in raw_features_categorical[feature_type]:
            # one-hot encode the categorical features
            new_features_df = pd.concat([new_features_df, one_hot_encode(split_df, feature)], axis=1)

    return new_features_df

In [2144]:
raw_features_df = extract_raw_features(split, "visits", visits_df)

# handle missing values
raw_features_df = raw_features_df.fillna(raw_features_df.median())

# save the raw features dataframe to a csv file
raw_features_df.to_csv(os.path.join(DATA_PATH, f"features_raw_{SPLIT_TYPE}_{SPLIT_GROUP}.csv"), index=False)
print(f"{len(raw_features_df.columns)} columns")
print(raw_features_df.columns)

29 columns
Index(['CSN', 'Visit_no', 'Age', 'Triage_Temp', 'Triage_HR', 'Triage_RR',
       'Triage_SpO2', 'Triage_SBP', 'Triage_DBP',
       'Race_American_Indian_or_Alaska_Native', 'Race_Asian',
       'Race_Black_or_African_American', 'Race_Declines_to_State',
       'Race_Native_Hawaiian_or_Other_Pacific_Islander', 'Race_Other',
       'Race_Unknown', 'Race_White', 'Ethnicity_Declines_to_State',
       'Ethnicity_Hispanic/Latino', 'Ethnicity_Non-Hispanic/Non-Latino',
       'Ethnicity_Unknown', 'Means_of_arrival_EMS',
       'Means_of_arrival_Other/Unknown', 'Means_of_arrival_Self', 'Gender_F',
       'Gender_M', 'Gender_U', 'Payor_class_Medicaid', 'Payor_class_Medicare'],
      dtype='object')


## Engineered features
- visit arrival time (cyclical)
- visit arrival season (categorical)
- shock index (hr bpm / SBP)
- Modified early warning score (MEWS)
- Total medication count (need to use dates for this)

In [2145]:
split_df = filter_by_split(split, visits_df)
engineered_features = split_df[['CSN', 'MRN', 'Arrival_time', 'Departure_time', 'Triage_HR', 'Triage_DBP', 'Triage_SBP', 'Triage_RR', 'Triage_Temp']].copy()

In [2146]:
# print the count of med rows with na entry dates
# Count the number of rows where Entry_date is NaN
na_entry_date_count = meds_df['Entry_date'].isna().sum()
na_start_date_count = meds_df['Start_date'].isna().sum()
na_end_date_count = meds_df['End_date'].isna().sum()
na_arrive_date_count = visits_df['Arrival_time'].isna().sum()
na_depart_date_count = visits_df['Departure_time'].isna().sum()
print(f"Number of rows with NaN Entry_date: {na_entry_date_count}")
print(f"Number of rows with NaN Start_date: {na_start_date_count}")
print(f"Number of rows with NaN End_date: {na_end_date_count}")
print(f"Number of rows with NaN Arrival_time: {na_arrive_date_count}")
print(f"Number of rows with NaN Departure_time: {na_depart_date_count}")


Number of rows with NaN Entry_date: 0
Number of rows with NaN Start_date: 155486
Number of rows with NaN End_date: 123111
Number of rows with NaN Arrival_time: 0
Number of rows with NaN Departure_time: 232


##### Total Medication Count

In [2147]:
from datetime import datetime
import time

def convert_datestr_epoch(datestr):
    """
    Convert a date string to epoch time without using pd.to_datetime, 
    handling ISO 8601 format and dates beyond pandas.Timestamp limits.
    
    :param datestr: The date string to convert (format: YYYY-MM-DDTHH:MM:SSZ)
    :return: The epoch time in seconds
    """
    #print(f"Converting date string: {datestr}")
    if pd.isna(datestr):
        return None
    try:
        # Parse the ISO 8601 date string into a datetime object
        dt = datetime.strptime(datestr, "%Y-%m-%dT%H:%M:%SZ")
        
        # Convert to epoch time in seconds
        epoch_time = int(time.mktime(dt.timetuple()))
        
        return epoch_time
    except ValueError:
        # Handle invalid date strings
        return None

In [2148]:
def get_meds_count(split_df, meds_df):
    """
    Get the count of current home meds for each visit.
    :param split_df: The dataframe with the visits
    :param meds_df: The dataframe with the meds
    :return: Dataframe with the count of home meds for each visit
    """
    # Create a copy to avoid modifying the original dataframe
    result_df = split_df.copy()
    
    # Create a new column for the count of home meds
    result_df["Home_meds_count"] = 0
    
    # Iterate through each row in the split dataframe
    for index, row in result_df.iterrows():
        # Get the MRN for the visit
        mrn = row["MRN"]
        arrival_time = convert_datestr_epoch(row["Arrival_time"])
        
        # Filter the meds dataframe for the MRN
        meds_for_mrn = meds_df[meds_df["MRN"] == mrn]
        
        # Count medications that were current at the time of the visit
        current_meds_count = 0
        for _, med_row in meds_for_mrn.iterrows():
            start_date = convert_datestr_epoch(med_row["Start_date"])
            end_date = convert_datestr_epoch(med_row["End_date"]) if pd.notna(med_row["End_date"]) else float('inf')
            
            # Check if medication was current during the visit
            # (started before arrival and either had no end date or ended after arrival)
            if start_date and start_date <= arrival_time and (not end_date or end_date > arrival_time):
                current_meds_count += 1
        
        # Set the count of home meds for this visit
        result_df.at[index, "Home_meds_count"] = current_meds_count
    
    return result_df

# Get the count of home meds for each visit
engineered_features = get_meds_count(engineered_features, meds_df)
engineered_features.drop(columns=["MRN"], inplace=True)
engineered_features.drop(columns=["Departure_time"], inplace=True)
engineered_features.columns

Index(['CSN', 'Arrival_time', 'Triage_HR', 'Triage_DBP', 'Triage_SBP',
       'Triage_RR', 'Triage_Temp', 'Home_meds_count'],
      dtype='object')

##### Arrival Time

In [2149]:
# lower year if it is greater than 2262
engineered_features['Arrival_time'] = engineered_features['Arrival_time'].str.replace(r'^.{4}', '2020', regex=True)
engineered_features['Arrival_time'] = pd.to_datetime(engineered_features['Arrival_time'])

# First calculate time in minutes since midnight
minutes_since_midnight = engineered_features['Arrival_time'].dt.hour * 60 + \
                         engineered_features['Arrival_time'].dt.minute


# Convert to radians (full circle = 2π)
time_in_rad = 2 * np.pi * minutes_since_midnight / (24 * 60)

# Create both features
engineered_features['arrival_time_sin'] = np.sin(time_in_rad)
engineered_features['arrival_time_cos'] = np.cos(time_in_rad)

print(engineered_features.columns)

Index(['CSN', 'Arrival_time', 'Triage_HR', 'Triage_DBP', 'Triage_SBP',
       'Triage_RR', 'Triage_Temp', 'Home_meds_count', 'arrival_time_sin',
       'arrival_time_cos'],
      dtype='object')


##### Arrival Season

In [2150]:
seasons = {
    1: "winter",
    2: "spring",
    3: "summer",
    4: "fall"
}
arrival_seasons = engineered_features['Arrival_time'].dt.month.map(lambda x: (x % 12 + 3) // 3)
# Convert to seasons
arrival_seasons = arrival_seasons.map(seasons)
# one-hot encode the seasons
arrival_seasons = pd.get_dummies(arrival_seasons, prefix="arrival_time_season").astype(int)
# concatenate the one-hot encoded seasons with the engineered features
engineered_features = pd.concat([engineered_features, arrival_seasons], axis=1)
engineered_features.drop(columns=['Arrival_time'], inplace=True)

engineered_features.columns

Index(['CSN', 'Triage_HR', 'Triage_DBP', 'Triage_SBP', 'Triage_RR',
       'Triage_Temp', 'Home_meds_count', 'arrival_time_sin',
       'arrival_time_cos', 'arrival_time_season_fall',
       'arrival_time_season_spring', 'arrival_time_season_summer',
       'arrival_time_season_winter'],
      dtype='object')

##### Shock Index

In [2151]:
engineered_features['shock_index'] = engineered_features['Triage_HR'] / engineered_features['Triage_DBP']
# cap at two decimal places
engineered_features['shock_index'] = engineered_features['shock_index'].round(2)
engineered_features['shock_index'] = engineered_features['shock_index'].replace([np.inf, -np.inf], np.nan)
engineered_features['shock_index'] = engineered_features['shock_index'].fillna(0)
engineered_features['shock_index'] = engineered_features['shock_index'].astype(float)

engineered_features.columns

Index(['CSN', 'Triage_HR', 'Triage_DBP', 'Triage_SBP', 'Triage_RR',
       'Triage_Temp', 'Home_meds_count', 'arrival_time_sin',
       'arrival_time_cos', 'arrival_time_season_fall',
       'arrival_time_season_spring', 'arrival_time_season_summer',
       'arrival_time_season_winter', 'shock_index'],
      dtype='object')

##### Modified early warning score (MEWS)

In [2152]:
def calculate_mews(sbp, hr, rr, temp, avpu=None):
    """
    Calculate Modified Early Warning Score (MEWS) from individual vital signs.
    
    Parameters:
    sbp (float): Systolic blood pressure in mmHg
    hr (float): Heart rate in beats per minute
    rr (float): Respiratory rate in breaths per minute
    temp (float): Temperature in Celsius
    avpu (str, optional): AVPU score ('A', 'V', 'P', 'U') if available
    
    Returns:
    int: The calculated MEWS score
    """
    # Initialize score
    mews_score = 0
    
    # Systolic BP scoring
    if sbp < 70:
        mews_score += 3
    elif 71 <= sbp <= 80:
        mews_score += 2
    elif 81 <= sbp <= 100:
        mews_score += 1
    elif sbp >= 200:
        mews_score += 2
    
    # Heart rate scoring
    if hr < 40:
        mews_score += 2
    elif 41 <= hr <= 50:
        mews_score += 1
    elif 101 <= hr <= 110:
        mews_score += 1
    elif 111 <= hr <= 129:
        mews_score += 2
    elif hr >= 130:
        mews_score += 3
    
    # Respiratory rate scoring
    if rr < 9:
        mews_score += 2
    elif 15 <= rr <= 20:
        mews_score += 1
    elif 21 <= rr <= 29:
        mews_score += 2
    elif rr >= 30:
        mews_score += 3
    
    # Temperature scoring
    if temp < 35.0:
        mews_score += 2
    elif temp >= 38.5:
        mews_score += 2
    
    # AVPU score if provided
    if avpu:
        if avpu == 'V':  # Responding to Voice
            mews_score += 1
        elif avpu == 'P':  # Responding to Pain
            mews_score += 2
        elif avpu == 'U':  # Unresponsive
            mews_score += 3
    
    return mews_score

In [2153]:
engineered_features['MEWS'] = engineered_features.apply(
    lambda row: calculate_mews(
        row['Triage_SBP'],
        row['Triage_HR'],
        row['Triage_RR'],
        row['Triage_Temp'],
        avpu=None  # Replace with actual AVPU data if available
    ),
    axis=1
)

engineered_features.drop(columns=['Triage_SBP', 'Triage_HR', 'Triage_RR', 'Triage_Temp', 'Triage_DBP'], inplace=True)
engineered_features.columns

Index(['CSN', 'Home_meds_count', 'arrival_time_sin', 'arrival_time_cos',
       'arrival_time_season_fall', 'arrival_time_season_spring',
       'arrival_time_season_summer', 'arrival_time_season_winter',
       'shock_index', 'MEWS'],
      dtype='object')

In [2154]:
# write the engineered features to a csv file
engineered_features.to_csv(os.path.join(DATA_PATH, f"features_engineered_{SPLIT_TYPE}_{SPLIT_GROUP}.csv"), index=False)

## Combining Raw + Engineered Features

In [2155]:
all_features = pd.merge(raw_features_df, engineered_features, on='CSN', how='inner')
#all_features = all_features.drop(columns=['CSN'])
# save the engineered features dataframe to a csv file
all_features.to_csv(os.path.join(DATA_PATH, f"features_all_{SPLIT_TYPE}_{SPLIT_GROUP}.csv"), index=False)
print(f"{len(all_features.columns)} columns")
print(all_features.columns)

38 columns
Index(['CSN', 'Visit_no', 'Age', 'Triage_Temp', 'Triage_HR', 'Triage_RR',
       'Triage_SpO2', 'Triage_SBP', 'Triage_DBP',
       'Race_American_Indian_or_Alaska_Native', 'Race_Asian',
       'Race_Black_or_African_American', 'Race_Declines_to_State',
       'Race_Native_Hawaiian_or_Other_Pacific_Islander', 'Race_Other',
       'Race_Unknown', 'Race_White', 'Ethnicity_Declines_to_State',
       'Ethnicity_Hispanic/Latino', 'Ethnicity_Non-Hispanic/Non-Latino',
       'Ethnicity_Unknown', 'Means_of_arrival_EMS',
       'Means_of_arrival_Other/Unknown', 'Means_of_arrival_Self', 'Gender_F',
       'Gender_M', 'Gender_U', 'Payor_class_Medicaid', 'Payor_class_Medicare',
       'Home_meds_count', 'arrival_time_sin', 'arrival_time_cos',
       'arrival_time_season_fall', 'arrival_time_season_spring',
       'arrival_time_season_summer', 'arrival_time_season_winter',
       'shock_index', 'MEWS'],
      dtype='object')


## Embedding features
- Visit diagnosis info
    - CC
- Meds
    - Name
    - Generic_name
    - Med_class
    - Med_subclass
- PMH
    - CodeType (only for prefix)
    - Code
    - Desc10
    - DescCCS

In [2156]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
from typing import Dict, List, Union, Optional, Tuple
import logging
from tqdm import tqdm

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class ClinicalTextEmbedder:
    """
    A modular class for generating embeddings from clinical text using different BERT models.
    
    Supports both PyTorch native models and TensorFlow converted models.
    """
    
    # Dictionary mapping model shortcuts to their Hugging Face identifiers and config
    MODEL_MAP = {
        'bio_clinical_bert': {'path': 'emilyalsentzer/Bio_ClinicalBERT', 'from_tf': False, 'from_flax': False},
        'pubmed_bert': {'path': 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext', 'from_tf': False, 'from_flax': False},
        'blue_bert': {'path': 'bionlp/bluebert_pubmed_uncased_L-24_H-1024_A-16', 'from_tf': False, 'from_flax': True},
        'bert_base': {'path': 'bert-base-uncased', 'from_tf': False, 'from_flax': False},
    }
    
    def __init__(self, model_name: str = 'bio_clinical_bert', device: Optional[str] = None):
        """
        Initialize the embedder with a specific BERT model.
        
        Args:
            model_name: Either a shortcut name from MODEL_MAP or a direct Hugging Face model identifier
            device: Device to use ('cpu', 'cuda', or None to automatically select)
        """
        # Determine device if not specified
        if device is None:
            self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        else:
            self.device = device
        
        # Resolve model info from shortcuts if needed
        if model_name in self.MODEL_MAP:
            self.model_info = self.MODEL_MAP[model_name]
            self.model_path = self.model_info['path']
            self.from_tf = self.model_info['from_tf']
            self.from_flax = self.model_info['from_flax']
        else:
            # If custom model path provided, assume it's not from TF
            self.model_path = model_name
            self.from_tf = False
            self.from_flax = False
            
        logger.info(f"Loading model: {self.model_path} on {self.device} (from_tf={self.from_tf}) (from_flax={self.from_flax})")
        
        # Load tokenizer and model
        try:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
            self.model = AutoModel.from_pretrained(self.model_path, from_tf=self.from_tf, from_flax=self.from_flax).to(self.device)
            self.model.eval()  # Set model to evaluation mode
        except Exception as e:
            logger.error(f"Failed to load model {self.model_path}: {e}")
            raise
    
    # The rest of the class remains the same...
    # (get_embedding and get_embeddings_batch methods stay unchanged)
    
    def get_embedding(self, text: str, pooling: str = 'cls', max_length: int = 512) -> np.ndarray:
        """
        Generate an embedding for a single text input.
        
        Args:
            text: The input text to embed
            pooling: Pooling strategy ('cls', 'mean', or 'max')
            max_length: Maximum token length (will be truncated if longer)
            
        Returns:
            numpy array embedding vector
        """
        # Handle empty text
        if not text or pd.isna(text):
            text = ""
            
        # Tokenize the text
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(self.device)
        
        # Get model output
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        # Different pooling strategies
        last_hidden_state = outputs.last_hidden_state
        
        if pooling == 'cls':
            # Use CLS token embedding
            embedding = last_hidden_state[:, 0, :].cpu().numpy()[0]
        elif pooling == 'mean':
            # Use mean of all token embeddings
            # Create attention mask to prevent considering padding tokens
            attention_mask = inputs['attention_mask']
            # Multiply by attention mask to zero out padding tokens
            embedding = torch.sum(last_hidden_state * attention_mask.unsqueeze(-1), 1) / torch.sum(attention_mask, 1, keepdim=True)
            embedding = embedding.cpu().numpy()[0]
        elif pooling == 'max':
            # Use max pooling
            # Set padding tokens to large negative number so they're never the max
            attention_mask = inputs['attention_mask']
            tokens_mask = attention_mask.unsqueeze(-1)
            masked_hidden = last_hidden_state * tokens_mask + (1 - tokens_mask) * -1e9
            embedding = torch.max(masked_hidden, dim=1)[0].cpu().numpy()[0]
        else:
            raise ValueError(f"Unsupported pooling strategy: {pooling}")
            
        return embedding
    
    def get_embeddings_batch(self, texts: List[str], batch_size: int = 8, 
                           pooling: str = 'cls', max_length: int = 512, 
                           show_progress: bool = True) -> np.ndarray:
        """
        Generate embeddings for a batch of texts.
        
        Args:
            texts: List of input texts
            batch_size: Batch size for processing
            pooling: Pooling strategy ('cls', 'mean', or 'max')
            max_length: Maximum token length
            show_progress: Whether to show a progress bar
            
        Returns:
            numpy array of embedding vectors with shape (len(texts), embedding_dim)
        """
        embeddings = []
        
        # Create batches
        iterator = range(0, len(texts), batch_size)
        if show_progress:
            iterator = tqdm(iterator, desc=f"Generating embeddings with {self.model_path}")
            
        for i in iterator:
            batch_texts = texts[i:i+batch_size]
            
            # Handle empty or NaN texts
            batch_texts = [text if text and not pd.isna(text) else "" for text in batch_texts]
            
            # Tokenize the batch
            inputs = self.tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length
            ).to(self.device)
            
            # Get model output
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            # Different pooling strategies
            last_hidden_state = outputs.last_hidden_state
            attention_mask = inputs['attention_mask']
            
            if pooling == 'cls':
                # Use CLS token embedding
                batch_embeddings = last_hidden_state[:, 0, :].cpu().numpy()
            elif pooling == 'mean':
                # Use mean of all token embeddings, properly masked
                sum_embeddings = torch.sum(last_hidden_state * attention_mask.unsqueeze(-1), 1)
                sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9).unsqueeze(-1)
                batch_embeddings = (sum_embeddings / sum_mask).cpu().numpy()
            elif pooling == 'max':
                # Use max pooling
                tokens_mask = attention_mask.unsqueeze(-1)
                masked_hidden = last_hidden_state * tokens_mask + (1 - tokens_mask) * -1e9
                batch_embeddings = torch.max(masked_hidden, dim=1)[0].cpu().numpy()
            else:
                raise ValueError(f"Unsupported pooling strategy: {pooling}")
                
            embeddings.extend(batch_embeddings)
            
        return np.array(embeddings)
   

##### Visit Embedding

In [2157]:
for model in ClinicalTextEmbedder.MODEL_MAP.keys():
    print(f"Generating visit embedding with model: {model}") 
    
    # Initialize embedder
    embedder = ClinicalTextEmbedder(model_name=model)

    # Filter the visits dataframe by the split
    split_visits_df = filter_by_split(split, visits_df)

    # Use the existing embedding_text column format
    split_visits_df['embedding_text'] = "Chief complaint: " + split_visits_df['CC'].astype(str) + ", "

    # Create a mapping to store CSN -> embedding for later use
    visit_embeddings = {}

    # Process in batches for efficiency
    BATCH_SIZE = 32  # Adjust based on available memory
    texts = split_visits_df['embedding_text'].tolist()
    csns = split_visits_df['CSN'].tolist()

    # Generate embeddings
    embeddings = embedder.get_embeddings_batch(
        texts, 
        batch_size=BATCH_SIZE,
        pooling='mean',  # Using mean pooling across tokens
        max_length=512,
        show_progress=True
    )

    # Create a dictionary mapping CSN to embedding
    for i, csn in enumerate(csns):
        visit_embeddings[csn] = embeddings[i]

    print(f"Generated {len(visit_embeddings)} visit embeddings")

    # Save embeddings to file
    import pickle
    import os

    # Create a directory for embeddings if it doesn't exist
    os.makedirs(os.path.join(DATA_PATH, "embeddings"), exist_ok=True)

    # Save as pickle file for easy loading
    embedding_file = os.path.join(DATA_PATH, f"embeddings/visit_diagnosis_{SPLIT_TYPE}_{SPLIT_GROUP}_{model}.pkl")
    with open(embedding_file, 'wb') as f:
        pickle.dump(visit_embeddings, f)

    print(f"Saved visit diagnosis embeddings to {embedding_file}")


Generating visit embedding with model: bio_clinical_bert


2025-04-17 20:50:06,190 - INFO - Loading model: emilyalsentzer/Bio_ClinicalBERT on cpu (from_tf=False) (from_flax=False)
Generating embeddings with emilyalsentzer/Bio_ClinicalBERT: 100%|██████████| 46/46 [00:10<00:00,  4.37it/s]
2025-04-17 20:50:19,483 - INFO - Loading model: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext on cpu (from_tf=False) (from_flax=False)


Generated 1447 visit embeddings
Saved visit diagnosis embeddings to ./data/embeddings/visit_diagnosis_random_test_bio_clinical_bert.pkl
Generating visit embedding with model: pubmed_bert


Generating embeddings with microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext: 100%|██████████| 46/46 [00:07<00:00,  6.20it/s]
2025-04-17 20:50:27,468 - INFO - Loading model: bionlp/bluebert_pubmed_uncased_L-24_H-1024_A-16 on cpu (from_tf=False) (from_flax=True)


Generated 1447 visit embeddings
Saved visit diagnosis embeddings to ./data/embeddings/visit_diagnosis_random_test_pubmed_bert.pkl
Generating visit embedding with model: blue_bert


All Flax model weights were used when initializing BertModel.

All the weights of BertModel were initialized from the Flax model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use BertModel for predictions without further training.
Generating embeddings with bionlp/bluebert_pubmed_uncased_L-24_H-1024_A-16: 100%|██████████| 46/46 [00:22<00:00,  2.01it/s]
2025-04-17 20:50:53,571 - INFO - Loading model: bert-base-uncased on cpu (from_tf=False) (from_flax=False)


Generated 1447 visit embeddings
Saved visit diagnosis embeddings to ./data/embeddings/visit_diagnosis_random_test_blue_bert.pkl
Generating visit embedding with model: bert_base


Generating embeddings with bert-base-uncased: 100%|██████████| 46/46 [00:07<00:00,  6.21it/s]

Generated 1447 visit embeddings
Saved visit diagnosis embeddings to ./data/embeddings/visit_diagnosis_random_test_bert_base.pkl


##### Home Medication Embedding

In [2158]:
# Link visits to their current medications
split_visits_df = filter_by_split(split, visits_df)

# Create a dictionary to store medications for each CSN
visit_meds = {}

# Process each visit
for index, row in split_visits_df.iterrows():
    csn = row["CSN"]
    mrn = row["MRN"]
    arrival_time = convert_datestr_epoch(row["Arrival_time"])
    
    # Filter the meds dataframe for the MRN
    meds_for_mrn = meds_df[meds_df["MRN"] == mrn]
    
    # Initialize list to store current meds for this visit
    current_meds = []
    
    # Check each medication
    for _, med_row in meds_for_mrn.iterrows():
        # start_date = convert_datestr_epoch(med_row["Start_date"])
        # end_date = convert_datestr_epoch(med_row["End_date"]) if pd.notna(med_row["End_date"]) else None
        entry_date = convert_datestr_epoch(med_row["Entry_date"])
        
        # Check if medication was current during the visit
        # (started before arrival and either had no end date or ended after arrival)
        #if start_date and start_date <= arrival_time and (not end_date or end_date > arrival_time):
        # I am choosing a broader criteria in case past medications are relevant
        if entry_date <= arrival_time:
            # Add this medication to the current_meds list
            current_meds.append({
                "Name": med_row["Name"],
                "Generic_name": med_row["Generic_name"],
                "Med_class": med_row["Med_class"],
                "Med_subclass": med_row["Med_subclass"],
            })
    
    # Store the list of medications for this CSN
    visit_meds[csn] = current_meds

# Create a DataFrame that lists each visit once with its medications as a list
med_visit_df = pd.DataFrame({
    "CSN": split_visits_df["CSN"],
    "MRN": split_visits_df["MRN"],
    "Medications": [visit_meds.get(csn, []) for csn in split_visits_df["CSN"]]
})

# Preview the results
print(f"Number of visits: {len(med_visit_df)}")
print("\nSample of visits with medication lists:")
print(med_visit_df[["CSN", "MRN", "Medications"]].head(3))

Number of visits: 1447

Sample of visits with medication lists:
        CSN       MRN                                        Medications
0  99603825  99430503  [{'Name': 'ACETAMINOPHEN 325 MG PO TABS', 'Gen...
1  99908562  99430503  [{'Name': 'ACETAMINOPHEN 325 MG PO TABS', 'Gen...
2  99892292  99430503  [{'Name': 'ACETAMINOPHEN 325 MG PO TABS', 'Gen...


In [2168]:
# Apply the conditional logic row-wise
def generate_med_embedding_text(meds):
    if meds and len(meds) > 0:  # Check if the 'Medications' field is not empty
        return "\n".join([f"Medication name: {med['Name']}, Generic name: {med['Generic_name']}, Class: {med['Med_class']}, Subclass: {med['Med_subclass']}" for med in meds])
    else:
        return ""

# Apply the function to the 'Medications' column
med_visit_df['embedding_text'] = med_visit_df['Medications'].apply(generate_med_embedding_text)

# Display the first row of the embedding_text column
print(med_visit_df['embedding_text'][0])

Medication name: ACETAMINOPHEN 325 MG PO TABS, Generic name: acetaminophen 325 mg tablet, Class: ANALGESIC/ANTIPYRETICS,NON-SALICYLATE, Subclass: Analgesic or Antipyretic Non-Opioid
Medication name: THERAPEUTIC MULTIVITAMIN PO TABS, Generic name: therapeutic multivitamin tablet, Class: MULTIVITAMIN PREPARATIONS, Subclass: Multivitamins
Medication name: FLUTICASONE PROPIONATE 50 MCG/ACTUATION NASAL SPSN, Generic name: fluticasone propionate 50 mcg/actuation nasal spray,suspension, Class: NASAL ANTI-INFLAMMATORY STEROIDS, Subclass: Nasal Corticosteroids
Medication name: VITAMIN D3 25 MCG (1,000 UNIT) PO TABS, Generic name: cholecalciferol (vitamin D3) 25 mcg (1,000 unit) tablet, Class: VITAMIN D PREPARATIONS, Subclass: Vitamins - D Derivatives
Medication name: VITAMIN D3 25 MCG (1,000 UNIT) PO TABS, Generic name: cholecalciferol (vitamin D3) 25 mcg (1,000 unit) tablet, Class: VITAMIN D PREPARATIONS, Subclass: Vitamins - D Derivatives
Medication name: TADALAFIL 10 MG PO TABS, Generic name

In [2160]:
for model in ClinicalTextEmbedder.MODEL_MAP.keys():
    print(f"Generating meds embedding with model: {model}") 
    
    # Initialize embedder
    embedder = ClinicalTextEmbedder(model_name=model)


    # Create a mapping to store CSN -> embedding for later use
    med_embeddings = {}

    # Process in batches for efficiency
    BATCH_SIZE = 32  # Adjust based on available memory
    texts = med_visit_df['embedding_text'].tolist()
    csns = med_visit_df['CSN'].tolist()

    # Generate embeddings
    embeddings = embedder.get_embeddings_batch(
        texts, 
        batch_size=BATCH_SIZE,
        pooling='mean',  # Using mean pooling across tokens
        max_length=512,
        show_progress=True
    )

    # Create a dictionary mapping CSN to embedding
    for i, csn in enumerate(csns):
        med_embeddings[csn] = embeddings[i]

    print(f"Generated {len(med_embeddings)} meds embeddings")

    # Save embeddings to file
    import pickle
    import os

    # Create a directory for embeddings if it doesn't exist
    os.makedirs(os.path.join(DATA_PATH, "embeddings"), exist_ok=True)

    # Save as pickle file for easy loading
    embedding_file = os.path.join(DATA_PATH, f"embeddings/meds_{SPLIT_TYPE}_{SPLIT_GROUP}_{model}.pkl")
    with open(embedding_file, 'wb') as f:
        pickle.dump(med_embeddings, f)

    print(f"Saved meds embeddings to {embedding_file}")


2025-04-17 20:51:02,113 - INFO - Loading model: emilyalsentzer/Bio_ClinicalBERT on cpu (from_tf=False) (from_flax=False)


Generating meds embedding with model: bio_clinical_bert


Generating embeddings with emilyalsentzer/Bio_ClinicalBERT: 100%|██████████| 46/46 [04:26<00:00,  5.79s/it]
2025-04-17 20:55:29,285 - INFO - Loading model: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext on cpu (from_tf=False) (from_flax=False)


Generated 1447 meds embeddings
Saved meds embeddings to ./data/embeddings/meds_random_test_bio_clinical_bert.pkl
Generating meds embedding with model: pubmed_bert


Generating embeddings with microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext: 100%|██████████| 46/46 [04:23<00:00,  5.73s/it]
2025-04-17 20:59:53,501 - INFO - Loading model: bionlp/bluebert_pubmed_uncased_L-24_H-1024_A-16 on cpu (from_tf=False) (from_flax=True)


Generated 1447 meds embeddings
Saved meds embeddings to ./data/embeddings/meds_random_test_pubmed_bert.pkl
Generating meds embedding with model: blue_bert


All Flax model weights were used when initializing BertModel.

All the weights of BertModel were initialized from the Flax model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use BertModel for predictions without further training.
Generating embeddings with bionlp/bluebert_pubmed_uncased_L-24_H-1024_A-16: 100%|██████████| 46/46 [16:28<00:00, 21.50s/it]
2025-04-17 21:16:25,413 - INFO - Loading model: bert-base-uncased on cpu (from_tf=False) (from_flax=False)


Generated 1447 meds embeddings
Saved meds embeddings to ./data/embeddings/meds_random_test_blue_bert.pkl
Generating meds embedding with model: bert_base


Generating embeddings with bert-base-uncased: 100%|██████████| 46/46 [04:44<00:00,  6.19s/it]

Generated 1447 meds embeddings
Saved meds embeddings to ./data/embeddings/meds_random_test_bert_base.pkl


##### Past Medical History Embedding

In [2161]:
# this is organized by MRN but the Noted_date can be used to link the pmh to the visit
# Link visits to their past medical history
split_visits_df = filter_by_split(split, visits_df)

# Create a dictionary to store medical history for each CSN
visit_pmh_hist = {}

# Process each visit
for index, row in split_visits_df.iterrows():
    csn = row["CSN"]
    mrn = row["MRN"]
    arrival_time = convert_datestr_epoch(row["Arrival_time"])
    
    # Filter the meds dataframe for the MRN
    pmh_for_mrn = pmh_df[pmh_df["MRN"] == mrn]
    
    # Initialize list to store current meds for this visit
    pmh = []
    
    # Check each medication
    for _, hist_row in pmh_for_mrn.iterrows():
        noted_date = convert_datestr_epoch(hist_row["Noted_date"])
        
        # Check if medical history cutoff was current during the visit
        if noted_date and noted_date <= arrival_time:
            # Add this history item to the medical_history list
            pmh.append({
                "CodeType": hist_row["CodeType"],
                "Code": hist_row["Code"],
                "Desc10": hist_row["Desc10"],
                "CCS": hist_row["CCS"],
                "DescCCS": hist_row["DescCCS"],
            })
    
    # Store the list of medications for this CSN
    visit_pmh_hist[csn] = pmh

# Create a DataFrame that lists each visit once with its medications as a list
pmh_visit_df = pd.DataFrame({
    "CSN": split_visits_df["CSN"],
    "MRN": split_visits_df["MRN"],
    "PMH": [visit_pmh_hist.get(csn, []) for csn in split_visits_df["CSN"]]
})

# Preview the results
print(f"Number of visits: {len(visit_pmh_hist)}")
print("\nSample of visits with history lists:")
print(pmh_visit_df[["CSN", "MRN", "PMH"]].head(3))

Number of visits: 1447

Sample of visits with history lists:
        CSN       MRN                                                PMH
0  99603825  99430503  [{'CodeType': 'Dx10', 'Code': 'Q059', 'Desc10'...
1  99908562  99430503  [{'CodeType': 'Dx10', 'Code': 'Q059', 'Desc10'...
2  99892292  99430503  [{'CodeType': 'Dx10', 'Code': 'Q059', 'Desc10'...


In [2167]:
# Apply the conditional logic row-wise
def generate_pmh_embedding_text(pmh):
    if pmh and len(pmh) > 0:  # Check if the 'Medications' field is not empty
        return "\n".join([f"Code Type: {hist['CodeType']}, Code: {hist['Code']}, Code Description: {hist['Desc10']}, CCS: {hist['CCS']}, CCS Description: {hist['DescCCS']}" for hist in pmh])
    else:
        return ""

# Apply the function to the 'Medications' column
pmh_visit_df['embedding_text'] = pmh_visit_df['PMH'].apply(generate_pmh_embedding_text)

# Display the first row of the embedding_text column
print(pmh_visit_df['embedding_text'][0])

Code Type: Dx10, Code: Q059, Code Description: Spina bifida, unspecified, CCS: 216.0, CCS Description: Nervous system congenital anomalies
Code Type: Dx10, Code: M869, Code Description: Osteomyelitis, unspecified, CCS: 201.0, CCS Description: Infective arthritis and osteomyelitis (except that caused by tuberculosis or sexually transmitted disease)
Code Type: Dx10, Code: N1330, Code Description: Unspecified hydronephrosis, CCS: 161.0, CCS Description: Other diseases of kidney and ureters
Code Type: Dx10, Code: N319, Code Description: Neuromuscular dysfunction of bladder, unspecified, CCS: 162.0, CCS Description: Other diseases of bladder and urethra
Code Type: Dx10, Code: E860, Code Description: Dehydration, CCS: 55.0, CCS Description: Fluid and electrolyte disorders
Code Type: Dx10, Code: R339, Code Description: Retention of urine, unspecified, CCS: 163.0, CCS Description: Genitourinary symptoms and ill-defined conditions
Code Type: Dx10, Code: N19, Code Description: Unspecified kidney

In [ ]:
for model in ClinicalTextEmbedder.MODEL_MAP.keys():
    print(f"Generating history embedding with model: {model}") 
    
    # Initialize embedder
    embedder = ClinicalTextEmbedder(model_name=model)


    # Create a mapping to store CSN -> embedding for later use
    pmh_embeddings = {}

    # Process in batches for efficiency
    BATCH_SIZE = 32  # Adjust based on available memory
    texts = pmh_visit_df['embedding_text'].tolist()
    csns = pmh_visit_df['CSN'].tolist()

    # Generate embeddings
    embeddings = embedder.get_embeddings_batch(
        texts, 
        batch_size=BATCH_SIZE,
        pooling='mean',  # Using mean pooling across tokens
        max_length=512,
        show_progress=True
    )

    # Create a dictionary mapping CSN to embedding
    for i, csn in enumerate(csns):
        pmh_embeddings[csn] = embeddings[i]

    print(f"Generated {len(pmh_embeddings)} pmh embeddings")

    # Save embeddings to file
    import pickle
    import os

    # Create a directory for embeddings if it doesn't exist
    os.makedirs(os.path.join(DATA_PATH, "embeddings"), exist_ok=True)

    # Save as pickle file for easy loading
    embedding_file = os.path.join(DATA_PATH, f"embeddings/pmh_{SPLIT_TYPE}_{SPLIT_GROUP}_{model}.pkl")
    with open(embedding_file, 'wb') as f:
        pickle.dump(med_embeddings, f)

    print(f"Saved pmh embeddings to {embedding_file}")

2025-04-17 21:21:15,635 - INFO - Loading model: emilyalsentzer/Bio_ClinicalBERT on cpu (from_tf=False) (from_flax=False)


Generating history embedding with model: bio_clinical_bert


Generating embeddings with emilyalsentzer/Bio_ClinicalBERT: 100%|██████████| 46/46 [05:25<00:00,  7.09s/it]
2025-04-17 21:26:42,658 - INFO - Loading model: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext on cpu (from_tf=False) (from_flax=False)


Generated 1447 pmh embeddings
Saved pmh embeddings to ./data/embeddings/pmh_random_test_bio_clinical_bert.pkl
Generating history embedding with model: pubmed_bert


Generating embeddings with microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext: 100%|██████████| 46/46 [05:16<00:00,  6.88s/it]
2025-04-17 21:31:59,870 - INFO - Loading model: bionlp/bluebert_pubmed_uncased_L-24_H-1024_A-16 on cpu (from_tf=False) (from_flax=True)


Generated 1447 pmh embeddings
Saved pmh embeddings to ./data/embeddings/pmh_random_test_pubmed_bert.pkl
Generating history embedding with model: blue_bert


All Flax model weights were used when initializing BertModel.

All the weights of BertModel were initialized from the Flax model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use BertModel for predictions without further training.
Generating embeddings with bionlp/bluebert_pubmed_uncased_L-24_H-1024_A-16: 100%|██████████| 46/46 [18:54<00:00, 24.67s/it]
2025-04-17 21:50:57,950 - INFO - Loading model: bert-base-uncased on cpu (from_tf=False) (from_flax=False)


Generated 1447 pmh embeddings
Saved pmh embeddings to ./data/embeddings/pmh_random_test_blue_bert.pkl
Generating history embedding with model: bert_base


Generating embeddings with bert-base-uncased: 100%|██████████| 46/46 [05:48<00:00,  7.58s/it]

Generated 1447 pmh embeddings
Saved pmh embeddings to ./data/embeddings/pmh_random_test_bert_base.pkl


## Target Feature

In [8]:
def isFastTracked(row):
    """
    Determine if a visit is fast tracked based on the Triage_Acuity and ED LOS.
    
    :param row: A row from the DataFrame
    :return: 1 if fast tracked, 0 otherwise
    """
    try:
        # Convert Triage_acuity to an integer
        acuity = int(row['Triage_acuity'][0])
        
        # Check fast track conditions
        if acuity in [1, 2]:
            return 1
        elif acuity == 3 and row['ED_LOS'] <= 1:
            return 1
        else:
            return 0
    except (ValueError, TypeError):
        # Handle invalid or missing data
        return 0

# Target features
# Acuity 1-2
# Acuity 3 with a threshold of ED LOS 1 hr or less where patient was discharged
features_target_df = pd.DataFrame()
split_visits_df = filter_by_split(split, visits_df)
features_target_df['CSN'] = split_visits_df['CSN']
features_target_df['FAST_TRACK'] = split_visits_df.apply(isFastTracked, axis=1)

# Write the target features to a CSV file
features_target_df.to_csv(os.path.join(DATA_PATH, f"features_target_{SPLIT_TYPE}_{SPLIT_GROUP}.csv"), index=False)